# Lesson 28 Lab — GPU Memory, Concurrency, and Cost Estimation

**Puzzle:** How many requests fit after INT4 weight compression, and which hidden assumptions can invalidate that number?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Capacity uses total/usable HBM, weight and scale bytes, runtime reserve, workspaces, KV per request, fragmentation, tensor parallelism, and traffic context distribution.

### Core mechanism

A first bound is `requests = floor((usable - weights - workspace) / KV_per_request)`. Cost per token then depends on hourly price divided by achieved, quality-approved tokens per hour.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "28-gpu-capacity-cost"
device = require_cuda()
torch.manual_seed(2026 + 28)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

INT4 ideal bytes may make weights fit while leaving no useful KV/concurrency margin. Multi-GPU sharding adds communication and changes both cost and latency.

### What this code tests

The lab seeds a 70B arithmetic model with live RTX 5090 memory but explicitly does not allocate or benchmark a 70B model.

**Experiment:** Read live free memory from the RTX GPU and build BF16 versus INT4 capacity projections for a 70B-class model without allocating the model.

**Declared evidence label:** `capacity-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
free,total=torch.cuda.mem_get_info(); cfg={"parameters":70_000_000_000,"layers":80,"kv_heads":8,"head_dim":128,"context":8192}
def plan(weight_bytes,cache_bytes):
    weights=cfg["parameters"]*weight_bytes; reserve=total*0.10; usable=max(0,total-reserve-weights); kv=2*cfg["layers"]*cfg["context"]*cfg["kv_heads"]*cfg["head_dim"]*cache_bytes
    return {"weight_gib":round(weights/2**30,3),"kv_per_request_gib":round(kv/2**30,3),"projected_requests":max(0,int(usable//kv)),"single_gpu_weight_fit":weights<total-reserve}
plans={"bf16_weights_bf16_kv":plan(2,2),"int4_ideal_weights_bf16_kv":plan(0.5,2),"int4_ideal_weights_int8_kv":plan(0.5,1)}
result=base_result(28,"capacity-model"); result.update({"live_memory":{"free_gib":round(free/2**30,3),"total_gib":round(total/2**30,3)},"assumptions":cfg,"plans":plans,
    "conclusion":"Arithmetic capacity projections used live GPU memory but did not claim that a 70B engine loaded or met latency SLOs."})


## 3. Inspect the evidence

Label the result as a capacity model. It cannot establish latency, model quality, or whether a particular 70B engine will load.

### Acceptance and rollback gate

Use ranges and safety margins, then validate with the actual engine's measured peak, sustained concurrency, SLO, utilization, and cloud billing unit.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "assumptions": {
    "context": 8192,
    "head_dim": 128,
    "kv_heads": 8,
    "layers": 80,
    "parameters": 70000000000
  },
  "conclusion": "Arithmetic capacity projections used live GPU memory but did not claim that a 70B engine loaded or met latency SLOs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "capacity-model",
  "executed_at_utc": "2026-08-07T14:46:23+00:00",
  "lesson": 28,
  "live_memory": {
    "free_gib": 30.863,
    "total_gib": 31.358
  },
  "plans": {
    "bf16_weights_bf16_kv": {
      "kv_per_request_gib": 2.5,
      "projected_requests": 0,
      "single_gpu_weight_fit": false,
      "weight_gib": 130.385
    },
    "int4_ideal_weights_bf16_kv": {
      "kv_per_request_gib": 2.5,
      "projected_requests": 0,
      "single_gpu_weight_fit": false,
      "weight_gib": 32.596
    },


## 4. Explain the result

Use ranges and safety margins, then validate the chosen point with the actual engine and traffic distribution.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).